# 第 2 天练习 —— 本地 Ollama 摘要网页

## 练习目标（理念）

用 **Ollama 的 OpenAI 兼容接口**（`/v1`）调用本地小模型 `llama3.2:1b`：抓取 Anthropic 官网正文，再让模型做简要摘要。

- **输入**：`fetch_website_contents("https://www.anthropic.com/")` 返回的网页文本
- **输出**：本地模型生成的简短摘要（`print`）
- **对比点**：不走云端 OpenAI，而是 `base_url=http://localhost:11434/v1`

## 和本课 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Ollama 拉取模型 | `!ollama pull llama3.2:1b` |
| OpenAI 兼容客户端 | `OpenAI(base_url=..., api_key='ollama')` |
| Chat Completions | `ollama.chat.completions.create(...)` |
| 网页抓取复用 | `from scraper import fetch_website_contents` |

## 怎么跑

1. 本机已安装并启动 Ollama；先跑「pull」单元格确保有 `llama3.2:1b`
2. 同目录有可用的 `scraper.py`（提供 `fetch_website_contents`）
3. 从上到下运行；可把 URL 换成别的站点再摘要

我的 fork：https://github.com/JaymanR/llm_engineering


In [ ]:
# ========== 导入 + 默认云端客户端（本练习后面主要用 Ollama）==========

# 从 openai 导入 OpenAI 客户端类：同一套 SDK 可指向云端或本地兼容网关
from openai import OpenAI

# 先建一个默认客户端（读环境变量 OPENAI_API_KEY）；本 notebook 摘要步骤实际用的是下面的 ollama 客户端
openai = OpenAI()


In [ ]:
# ========== 准备本地模型：用 shell 魔法拉取 Ollama 模型 ==========

# Jupyter 的 ! 会把这一行交给系统 shell 执行（不是 Python）
# ollama pull：从 Ollama 仓库下载模型权重；名字必须和后面 create(model=...) 一致
!ollama pull llama3.2:1b


In [ ]:
# ========== 指向本地 Ollama 的 OpenAI 兼容客户端 ==========

# Ollama 提供的 OpenAI 兼容基址：注意带 /v1；默认端口 11434
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# base_url 改写请求目标；api_key 对本地 Ollama 通常任意非空即可（这里用 'ollama'）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== 抓网页 → 拼 messages → 本地 llama3.2:1b 摘要 ==========

# 从同目录 scraper 模块导入抓取函数：返回网页正文文本
from scraper import fetch_website_contents

# system prompt 保持英文：定助手角色（发给模型的指令不翻译）
system_prompt = "you are a helpful ai assistant."
# user 前半段：任务说明（后面会再拼接 website 正文）
user_prompt = """
Please give a brief summarization of the following website. 
"""

# 抓取 Anthropic 官网正文；URL 字符串保持原样
website = fetch_website_contents("https://www.anthropic.com/")

# 组装 Chat Completions 的 messages：user 内容 = 任务说明 + 网页全文
messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + website}]

# 走本地 ollama 客户端；model id 必须与已 pull 的名字一致
response = ollama.chat.completions.create(model="llama3.2:1b", messages=messages)
# 取出助手回复正文并打印到标准输出
print(response.choices[0].message.content)
